In [ ]:
from google.colab import drive
import os

# This will prompt you for permission to access your Drive
drive.mount('/content/drive')

In [ ]:
# 1. Delete any old extracted data folders to start fresh
!rm -rf /content/data

# 2. Re-create the empty folder
!mkdir -p /content/data

# 3. Now perform the extraction only once
!tar -xf "/content/drive/MyDrive/MajorProject/BraTS2021_Training_Data.tar" -C /content/data

# 4. Verify the extraction by listing the contents
!ls /content/data

In [ ]:
!pip install -qU "monai[nibabel, tqdm]"

In [ ]:
import os
import glob
import torch
import torch.optim as optim
from google.colab import drive
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Spacingd,
    Orientationd, NormalizeIntensityd, RandCropByPosNegLabeld, ToTensord, MapTransform
)
from monai.data import Dataset, DataLoader
from monai.networks.nets import SegResNet
from monai.losses import DiceLoss

drive.mount('/content/drive')

In [ ]:
import os
import glob
import torch
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Spacingd, 
    Orientationd, NormalizeIntensityd, RandCropByPosNegLabeld, ToTensord, MapTransform
)
from monai.data import CacheDataset, DataLoader

# 1. Path verified from your successful extraction
data_dir = "/content/data" 

# 2. Collect and sort all patient files
images_flair = sorted(glob.glob(os.path.join(data_dir, "*", "*_flair.nii.gz")))
images_t1 = sorted(glob.glob(os.path.join(data_dir, "*", "*_t1.nii.gz")))
images_t1ce = sorted(glob.glob(os.path.join(data_dir, "*", "*_t1ce.nii.gz")))
images_t2 = sorted(glob.glob(os.path.join(data_dir, "*", "*_t2.nii.gz")))
labels = sorted(glob.glob(os.path.join(data_dir, "*", "*_seg.nii.gz")))

data_dicts = [
    {"image": [f, t1, tc, t2], "label": s} 
    for f, t1, tc, t2, s in zip(images_flair, images_t1, images_t1ce, images_t2, labels)
]

# 3. BraTS Multi-Channel conversion logic
class ConvertToMultiChannelBratsd(MapTransform):
    def __call__(self, data):
        d = dict(data)
        for key in self.keys:
            result = [
                torch.logical_or(d[key] == 1, d[key] == 4),
                torch.logical_or(torch.logical_or(d[key] == 1, d[key] == 4), d[key] == 2),
                d[key] == 4
            ]
            d[key] = torch.stack(result, axis=0).float()
        return d

# ... (Keep steps 1, 2, and 3 exactly the same) ...

# 4. Define the Pipeline
train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image"]),
    ConvertToMultiChannelBratsd(keys=["label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    RandCropByPosNegLabeld(
        keys=["image", "label"], label_key="label", spatial_size=[128, 128, 128], 
        pos=1, neg=1, 
        num_samples=1  # CHANGE 1: Reduce from 4 to 1 to speed up training
    ),
    ToTensord(keys=["image", "label"]),
])

# 5. Initialize CacheDataset
train_ds = CacheDataset(
    data=data_dicts[:500], # CHANGE 2: Reduce from 1000 to 500 cases for faster epochs
    transform=train_transforms, 
    cache_rate=0.02, 
    num_workers=4
)

# 6. Final DataLoader
train_loader = DataLoader(
    train_ds, 
    batch_size=1, 
    shuffle=True, 
    num_workers=2
)

print(f"Verified: {len(data_dicts[:500])} cases used. CacheDataset warming up...")

In [ ]:
device = torch.device("cuda")
model = SegResNet(spatial_dims=3, init_filters=16, in_channels=4, out_channels=3).to(device)

loss_function = DiceLoss(smooth_nr=0, smooth_dr=1e-5, squared_pred=True, to_onehot_y=False, sigmoid=True)
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

# Path to save progress in Drive
checkpoint_path = "/content/drive/MyDrive/MajorProject/segresnet_best.pth"
start_epoch = 0

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch']
    print(f"Resumed from Epoch {start_epoch}")

In [ ]:
import torch

# --- Configuration for 60 Epochs ---
num_epochs = 60  
checkpoint_path = "/content/drive/MyDrive/MajorProject/segresnet_best.pth"

print(f"Starting training from epoch {start_epoch} to {num_epochs}...")

for epoch in range(start_epoch, num_epochs):
    model.train()
    epoch_loss = 0
    
    # Progress through the 1,000 patient cases
    for step, batch_data in enumerate(train_loader):
        inputs, labels = batch_data["image"].to(device), batch_data["label"].to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
        # Print status every 50 steps to monitor progress
        if (step + 1) % 50 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{step+1}/{len(train_loader)}], Loss: {loss.item():.4f}")

    avg_loss = epoch_loss / len(train_loader)
    print(f"--- Finished Epoch {epoch+1} | Average Loss: {avg_loss:.4f} ---")

    # --- COUNTER SAVER ---
    # This ensures that even if Colab crashes, your work is safe in Drive
    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': avg_loss,
    }, checkpoint_path)
    
    print(f"Checkpoint successfully saved to Google Drive at Epoch {epoch+1}")